### 练习 1：搭建一个标准的 PINN 多层感知机（MLP）

In [8]:

import torch
import torch.nn as nn


class SimpleMLP(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=50, output_dim=1, num_layers=3):
        super().__init__()
        layers = []
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.Tanh())
        for _ in range(num_layers - 2):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.Tanh())
        layers.append(nn.Linear(hidden_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


model = SimpleMLP(input_dim=2, hidden_dim=50, output_dim=1, num_layers=3)
total_params = sum(p.numel() for p in model.parameters())
print(f"总参数数: {total_params}")  # 2751

总参数数: 2751


### 练习 2：体验高精度自动微分（计算一、二阶导数）

In [7]:

# 创建一个需要求导的输入
x = torch.randn(100, 2, requires_grad=True)
u = model(x)

# 计算一阶导数：∂u/∂x₁
grad_u_x1 = torch.autograd.grad(
    u, x,
    grad_outputs=torch.ones_like(u),
    create_graph=True,
    retain_graph=True
)[0]

# 计算二阶导数：∂²u/∂x₁²
grad2_u_x1x1 = torch.autograd.grad(
    grad_u_x1[:, 0:1], x,
    grad_outputs=torch.ones_like(grad_u_x1[:, 0:1]),
    create_graph=True
)[0]

print(f"u shape: {u.shape}")
print(f"∂u/∂x₁ shape: {grad_u_x1.shape}")
print(f"∂²u/∂x₁² shape: {grad2_u_x1x1.shape}")

u shape: torch.Size([100, 1])
∂u/∂x₁ shape: torch.Size([100, 2])
∂²u/∂x₁² shape: torch.Size([100, 2])


### 练习 3：双阶段优化器（Adam + L-BFGS）骨架模板

In [11]:
# Stage 1: Adam（快速下降）
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(5000):
    loss = compute_loss(model, data)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Stage 2: L-BFGS（精细收敛）
optimizer = torch.optim.LBFGS(model.parameters(), lr=1e-1)

def closure():
    optimizer.zero_grad()
    loss = compute_loss(model, data)
    loss.backward()
    return loss

for epoch in range(500):
    loss = optimizer.step(closure)

NameError: name 'data' is not defined